# 02 — Temporal EDA

## Objective
Understand activity, fraud prevalence, transaction value, and entity evolution across time; identify bursts and regime shifts that motivate temporal modeling.

> **Scientific contract:** fraud labels are validation ground truth, never unsupervised predictors. Chronological splits protect against future leakage. The test period is locked until Notebook 11.

### Outputs
Reproducible artifacts are saved to `artifacts/` and audit evidence to `reports/`. Interactive controls are for investigation/exploration; the underlying tables remain reproducible.

## 1. Build daily operating metrics

In [1]:
from pathlib import Path
import sys, json, warnings, numpy as np, pandas as pd
import plotly.express as px
from IPython.display import display, Markdown
import ipywidgets as widgets
warnings.filterwarnings("ignore")
ROOT=Path.cwd()
if not (ROOT/"data").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
from pipeline_utils import *
ART=ROOT/"artifacts"; REP=ROOT/"reports"; ART.mkdir(exist_ok=True); REP.mkdir(exist_ok=True)
RANDOM_STATE=42
fraud,_=load_data(); x=add_temporal_raw(fraud); daily=x.assign(day=x.purchase_time.dt.floor("D")).groupby("day").agg(transactions=("user_id","size"),fraud_rate=("class","mean"),revenue=("purchase_value","sum"),users=("user_id","nunique"),devices=("device_id","nunique"),ips=("ip_address","nunique")).reset_index(); display(daily.head())

,day,transactions,fraud_rate,revenue,users,devices,ips
0,2015-01-01 00:00:00+00:00,571,0.998249,19411,571,55,55
1,2015-01-02 00:00:00+00:00,736,0.985054,27412,736,85,85
2,2015-01-03 00:00:00+00:00,558,0.982079,20503,558,67,67
3,2015-01-04 00:00:00+00:00,639,0.965571,21311,639,84,84
4,2015-01-05 00:00:00+00:00,578,0.961938,21589,578,78,78


## 2. Interactive time grain and metric

In [2]:
grain=widgets.Dropdown(options=["D","W","M"],value="D",description="Grain"); metric=widgets.Dropdown(options=["transactions","fraud_rate","revenue","users","devices","ips"],value="fraud_rate",description="Metric"); out=widgets.Output()
def draw(*_):
    g=x.assign(period=x.purchase_time.dt.to_period({"D":"D","W":"W","M":"M"}[grain.value]).dt.start_time).groupby("period").agg(transactions=("user_id","size"),fraud_rate=("class","mean"),revenue=("purchase_value","sum"),users=("user_id","nunique"),devices=("device_id","nunique"),ips=("ip_address","nunique")).reset_index()
    with out: out.clear_output(); px.line(g,x="period",y=metric.value,title=f"{metric.value} by {grain.value}").show()
grain.observe(draw,'value'); metric.observe(draw,'value'); display(widgets.HBox([grain,metric]),out); draw()

Output()

## 3. Distribution diagnostics

In [3]:
px.box(x,x="class",y="purchase_value",points=False,title="Purchase value by ground truth").update_yaxes(type="log").show(); px.histogram(x,x="account_age_hours",color="class",nbins=80,barmode="overlay",title="Account age distribution").show()